# 🧠 LOGOS — GPU Training (Kaggle T4/P100)
**Vedic GEMM + Langevin Dynamics Physics Optimizer**

| Config | Value |
|--------|-------|
| Optimizer | Langevin Dynamics (NO Adam) |
| GEMM | Vedic Urdhva-Tiryagbhyam |
| Dataset | TinyStories ~50MB text |
| GPU | T4 / P100 16GB |
| d_model | 128 |
| Layers | 4 |
| Heads | 8 |
| Epochs | 5 |

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 1 — GPU + Environment Check
# ═══════════════════════════════════════════════════════
import os, subprocess, re, math, shutil, time, glob, json
import numpy as np
import matplotlib.pyplot as plt

WORK_DIR   = '/kaggle/working/LOGOS'
LOG_FILE   = '/kaggle/working/training_log.txt'
OUT_DIR    = '/kaggle/working/logos_trained'
TRAIN_FILE = '/kaggle/working/dataset.txt'

print('=== GPU ===')
os.system('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader')
print('\n=== CUDA ===')
os.system('nvcc --version | grep release')
print('\n=== GCC ===')
os.system('g++ --version | head -1')
print('\n=== CMake ===')
os.system('cmake --version | head -1')
print('\n=== CPU/RAM/Disk ===')
os.system('nproc && free -h | head -2 && df -h /kaggle/working')
print('\n✅ Cell 1 done')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 2 — Dataset: TinyStories 50MB
# Strategy:
#   1. Kaggle dataset se uthao (agar attached hai)
#   2. HuggingFace se wget karo
#   3. Synthetic fallback
# ═══════════════════════════════════════════════════════
TARGET_MB = 50
TARGET_BYTES = TARGET_MB * 1024 * 1024

def try_local_dataset():
    """Kaggle input se koi bhi large text file lo"""
    # .txt files
    for f in glob.glob('/kaggle/input/**/*.txt', recursive=True):
        sz = os.path.getsize(f)
        if sz > 5 * 1024 * 1024:
            print(f'Found local: {f} ({sz//1024//1024} MB)')
            return f, sz
    # .jsonl / .json files — extract text field
    for f in glob.glob('/kaggle/input/**/*.jsonl', recursive=True):
        sz = os.path.getsize(f)
        if sz > 5 * 1024 * 1024:
            print(f'Found JSONL: {f} ({sz//1024//1024} MB)')
            return f, sz
    return None, 0

def extract_text_jsonl(src, dst, max_bytes):
    written = 0
    with open(dst, 'w', encoding='utf-8') as out:
        with open(src, 'r', encoding='utf-8', errors='ignore') as inp:
            for line in inp:
                if written >= max_bytes: break
                try:
                    obj = json.loads(line)
                    text = obj.get('story', obj.get('text', obj.get('content','')))
                    if text:
                        chunk = text.strip() + '\n\n'
                        out.write(chunk)
                        written += len(chunk)
                except: pass
    return written

# ── Main logic ────────────────────────────────────────
if os.path.exists(TRAIN_FILE) and os.path.getsize(TRAIN_FILE) >= TARGET_BYTES:
    sz = os.path.getsize(TRAIN_FILE)//1024//1024
    print(f'✅ dataset.txt already exists: {sz} MB — skipping')
else:
    print(f'Preparing {TARGET_MB}MB dataset...')
    done = False

    # 1. Local Kaggle input
    local_f, local_sz = try_local_dataset()
    if local_f:
        if local_f.endswith('.jsonl'):
            extract_text_jsonl(local_f, TRAIN_FILE, TARGET_BYTES)
        else:
            # Direct copy / truncate
            with open(local_f, 'r', errors='ignore') as fin, open(TRAIN_FILE, 'w') as fout:
                fout.write(fin.read(TARGET_BYTES))
        done = True

    # 2. HuggingFace wget (needs internet on Kaggle)
    if not done:
        print('Trying HuggingFace download (~52MB)...')
        # TinyStoriesV2 GPT4 train — approx 2.1GB full, take first 52MB
        ret = subprocess.run([
            'wget', '-q', '--timeout=120',
            'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt',
            '-O', '/kaggle/working/tinystories_full.txt'
        ], capture_output=True)
        if ret.returncode == 0:
            full_sz = os.path.getsize('/kaggle/working/tinystories_full.txt')
            print(f'Downloaded: {full_sz//1024//1024} MB')
            # Take exactly 50MB
            with open('/kaggle/working/tinystories_full.txt','r',errors='ignore') as fin, \
                 open(TRAIN_FILE,'w') as fout:
                fout.write(fin.read(TARGET_BYTES))
            os.remove('/kaggle/working/tinystories_full.txt')
            done = True
        else:
            print(f'wget failed: {ret.stderr.decode()[:200]}')

    # 3. Synthetic fallback (2MB — for smoke test only)
    if not done:
        print('⚠️  Using synthetic dataset (2MB — enable internet on Kaggle for real data)')
        templates = [
            'Once upon a time there was a little girl named Lily. She loved to play in the garden with her dog Max.\n',
            'Tom was a curious boy who loved reading books about science and nature every evening.\n',
            'The sun rose slowly over the small village. Birds began to sing their morning songs.\n',
            'Anna had a red bicycle. She rode it to school every day and always arrived on time.\n',
            'Ben and his sister Sara found a tiny puppy near the old oak tree in the park.\n',
            'The old wizard lived in a tall tower. He spent his days studying ancient maps and books.\n',
            'Emma wanted to bake a cake for her mother birthday. She mixed flour, eggs and sugar carefully.\n',
            'Jake loved to draw pictures of dragons and castles in his notebook during class.\n',
        ]
        with open(TRAIN_FILE, 'w') as f:
            written = 0
            while written < 2 * 1024 * 1024:
                for t in templates:
                    f.write(t); written += len(t)

sz = os.path.getsize(TRAIN_FILE)//1024//1024
print(f'\n✅ Dataset ready: {sz} MB ({os.path.getsize(TRAIN_FILE):,} bytes)')
print('Preview (first 400 chars):')
print('-'*50)
with open(TRAIN_FILE) as f: print(f.read(400))
print('-'*50)

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 3 — Clone / Pull LOGOS
# ═══════════════════════════════════════════════════════
REPO_URL = 'https://github.com/Vikas8719/LOGOS.git'

if not os.path.exists(WORK_DIR):
    print('Cloning LOGOS...')
    ret = os.system(f'git clone {REPO_URL} {WORK_DIR}')
    print('✅ Clone done' if ret==0 else '❌ Clone failed')
else:
    print('Pulling latest...')
    ret = os.system(f'git -C {WORK_DIR} pull origin main')
    print('✅ Pull done' if ret==0 else '❌ Pull failed')

os.chdir(WORK_DIR)
print('\nLatest commit:')
os.system('git log --oneline -3')
print('\nFiles:')
os.system('ls src/ cuda/')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 4 — Build CPU + GPU
# Auto-detects GPU arch (T4=75, P100=60, A100=80)
# ═══════════════════════════════════════════════════════
os.chdir(WORK_DIR)
os.system('rm -rf build && mkdir build')

# Detect GPU compute capability
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    cap = result.stdout.strip().replace('.','')  # e.g. "7.5" → "75"
    print(f'GPU compute capability: sm_{cap}')
except:
    cap = '75'  # T4 default
    print(f'Could not detect GPU cap, using sm_{cap}')

cmake_cmd = f"""
cd {WORK_DIR} && cmake -B build \\
  -DCMAKE_BUILD_TYPE=Release \\
  -DCMAKE_CXX_FLAGS="-O3 -march=native -std=c++20" \\
  -DCMAKE_CUDA_ARCHITECTURES="{cap}" \\
  2>&1 | tail -8
"""
print('=== CMake Configure ===')
os.system(cmake_cmd)

print('\n=== Build ===')
ret = os.system(f'cd {WORK_DIR} && cmake --build build --parallel $(nproc) 2>&1')

print('\n=== Binaries ===')
os.system(f'ls -lh {WORK_DIR}/build/logos* 2>/dev/null || echo "No binary found"')

GPU_BIN = f'{WORK_DIR}/build/logos_gpu'
CPU_BIN = f'{WORK_DIR}/build/logos'
BINARY  = GPU_BIN if os.path.exists(GPU_BIN) else CPU_BIN
MODE    = '⚡ GPU (CUDA)' if os.path.exists(GPU_BIN) else '🐌 CPU fallback'
print(f'\nWill train with: {BINARY}  [{MODE}]')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 5 — Sanity Tests
# ═══════════════════════════════════════════════════════
os.chdir(WORK_DIR)
CPU_BIN = f'{WORK_DIR}/build/logos'

print('=== Unit Tests ===')
os.system(f'{CPU_BIN} --test 2>&1')

print('\n=== Forward Pass ===')
os.system(f'{CPU_BIN} --forward 2>&1')

print('\n=== VedicGEMM Benchmark ===')
os.system(f'{CPU_BIN} --benchmark 2>&1')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 6 — Patch train params for 50MB dataset
# main.cpp mein d_model=128, L=4, epochs=5 set karo
# (runtime override — source edit karke rebuild)
# ═══════════════════════════════════════════════════════
import re as _re

MAIN_CPP = f'{WORK_DIR}/src/main.cpp'

with open(MAIN_CPP, 'r') as f:
    src = f.read()

original = src

# d_model 64 → 128 — scope the match to the training model block so reruns are safe
model_block_pattern = r'(?s)(cfg\.vocab_size\s*=\s*tok\.vocab_size;.*?auto params = model\.parameters\(\);)'
model_block_match = _re.search(model_block_pattern, src)
if not model_block_match:
    raise RuntimeError('Could not locate the training model configuration block in main.cpp')
model_block = model_block_match.group(1)
d_model_pattern = r'(?m)^(\s*cfg\.d_model\s*=\s*)(64|128)(?=\s*;)'
d_model_values = list(_re.finditer(d_model_pattern, model_block))
if len(d_model_values) != 1:
    raise RuntimeError(
        f'Expected exactly one d_model assignment in the training model block; found {len(d_model_values)}'
    )
if d_model_values[0].group(2) == '64':
    updated_block = _re.sub(d_model_pattern, r'\g<1>128', model_block, count=1)
    src = src[:model_block_match.start(1)] + updated_block + src[model_block_match.end(1):]
# num_layers 2 → 4
src = _re.sub(r'(cfg\.num_layers\s*=\s*)2(;\s*//.*layers)', r'\g<1>4\2', src)
# num_heads 4 → 8
src = _re.sub(r'(cfg\.num_heads\s*=\s*)4;', r'\g<1>8;', src)
# SEQ 64 → 128
src = _re.sub(r'(int SEQ\s*=\s*)64;', r'\g<1>128;', src)
# EPOCHS 20 → 5
src = _re.sub(r'(int EPOCHS\s*=\s*)20;', r'\g<1>5;', src)
# LR 3e-4 → 2e-4 (stable for bigger model)
src = _re.sub(r'LangevinOptimizer langevin\(3e-4f', 'LangevinOptimizer langevin(2e-4f', src)

changed = src != original
with open(MAIN_CPP, 'w') as f:
    f.write(src)

print('Patched main.cpp:' if changed else 'No changes (already patched):')
display_terms = ('d_model', 'num_layers', 'num_heads', 'int SEQ', 'int EPOCHS', 'LangevinOptimizer langevin')
with open(MAIN_CPP, 'r') as f:
    for line_no, line in enumerate(f, 1):
        if any(term in line for term in display_terms) and '//' not in line:
            print(f'{line_no}: {line.rstrip()}')

# Rebuild with new params
print('\n=== Rebuilding ===')
build_dir = os.path.join(WORK_DIR, 'build')
cache_file = os.path.join(build_dir, 'CMakeCache.txt')
if not os.path.exists(cache_file):
    print('CMake build directory is not configured; configuring it now...')
    configure = subprocess.run(
        ['cmake', '-S', WORK_DIR, '-B', build_dir, '-DCMAKE_BUILD_TYPE=Release'],
        text=True, capture_output=True
    )
    if configure.returncode != 0:
        print(configure.stdout)
        print(configure.stderr)
        raise RuntimeError(f'CMake configuration failed with exit code {configure.returncode}')

build = subprocess.run(
    ['cmake', '--build', build_dir, '--parallel', str(os.cpu_count() or 1)],
    cwd=WORK_DIR, text=True, capture_output=True
)
build_output = (build.stdout + '\n' + build.stderr).strip()
if build_output:
    print('\n'.join(build_output.splitlines()[-8:]))
if build.returncode != 0:
    raise RuntimeError(f'Rebuild failed with exit code {build.returncode}')
print('✅ Rebuild done')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 7 — 🚀 GPU TRAINING (Main Run)
# Model  : d=128, L=4, H=8  (~7M params)
# Data   : 50MB TinyStories
# Optim  : Langevin Dynamics (Physics)
# Expected loss trajectory: 8.x → 6.x → 4.x → 3.x
# ═══════════════════════════════════════════════════════
os.chdir(WORK_DIR)

GPU_BIN = f'{WORK_DIR}/build/logos_gpu'
CPU_BIN = f'{WORK_DIR}/build/logos'
BINARY  = GPU_BIN if os.path.exists(GPU_BIN) else CPU_BIN
MODE    = '⚡ GPU (CUDA T4)' if os.path.exists(GPU_BIN) else '🐌 CPU'

ds_mb = os.path.getsize(TRAIN_FILE)//1024//1024
if BINARY == CPU_BIN and ds_mb >= 10:
    raise RuntimeError(
        f'CPU fallback refused for {ds_mb}MB dataset: Cell 7 is intended for GPU training and would likely exceed the session timeout.'
    )

print(f'╔══════════════════════════════════════════╗')
print(f'║    LOGOS GPU TRAINING                    ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  Mode    : {MODE:<30}║')
print(f'║  Dataset : {ds_mb} MB TinyStories{" "*(20-len(str(ds_mb)))}║')
print(f'║  Model   : d=128 | L=4 | H=8 | ~7M p   ║')
print(f'║  Optim   : Langevin Dynamics (Physics)  ║')
print(f'║  Eq      : dW = -γ∇L·dt + √(2γT)·η    ║')
print(f'╚══════════════════════════════════════════╝')
print()

steps_log, losses_log, smooth_log = [], [], []
smooth = -1
start_t = time.time()

process = subprocess.Popen(
    [BINARY, '--train', TRAIN_FILE],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
    cwd=WORK_DIR
)

os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)
with open(LOG_FILE, 'w') as log:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()

            m = _re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                s = int(m.group(1))
                l = float(m.group(2))
                if not (math.isnan(l) or math.isinf(l) or l > 50):
                    steps_log.append(s)
                    losses_log.append(l)
                    smooth = l if smooth < 0 else 0.95*smooth + 0.05*l
                    smooth_log.append(smooth)
    except KeyboardInterrupt:
        process.terminate()
        print('\n⏹️  Stopped by user — checkpoint saved above')

process.wait()
elapsed = time.time() - start_t

print(f'\n{"="*45}')
print(f'Training time : {elapsed/60:.1f} min')
if losses_log:
    drop = losses_log[0] - losses_log[-1]
    print(f'Start loss    : {losses_log[0]:.4f}  (PPL={math.exp(min(losses_log[0],12)):.0f})')
    print(f'Final loss    : {losses_log[-1]:.4f}  (PPL={math.exp(min(losses_log[-1],12)):.0f})')
    print(f'Best loss     : {min(losses_log):.4f}  (PPL={math.exp(min(min(losses_log),12)):.0f})')
    print(f'Total drop    : {drop:.4f} nats')
    print(f'Steps done    : {steps_log[-1]}')
    if drop > 1.0:
        print('✅ Model IS learning! Langevin works!')
    elif drop > 0.3:
        print('⚠️  Slight learning — more epochs needed')
    else:
        print('❌ Loss not dropping — check output above')
else:
    print('No loss data parsed')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 8 — Loss Curves (3 plots)
# ═══════════════════════════════════════════════════════
# Re-parse log in case cell 7 was interrupted
if not steps_log and os.path.exists(LOG_FILE):
    smooth = -1
    with open(LOG_FILE) as f:
        for line in f:
            m = _re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                s, l = int(m.group(1)), float(m.group(2))
                if not (math.isnan(l) or math.isinf(l) or l > 50):
                    steps_log.append(s)
                    losses_log.append(l)
                    smooth = l if smooth<0 else 0.95*smooth + 0.05*l
                    smooth_log.append(smooth)

if len(steps_log) < 3:
    print('Not enough data — run Cell 7 first')
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('LOGOS — Langevin Dynamics Training (50MB TinyStories)', fontsize=13, fontweight='bold')

    # Plot 1: Raw + Smoothed loss
    axes[0].plot(steps_log, losses_log, color='steelblue', lw=0.6, alpha=0.4, label='Raw loss')
    axes[0].plot(steps_log, smooth_log, color='crimson',   lw=2.0, label='EMA smoothed')
    axes[0].axhline(losses_log[0],  color='orange', ls='--', lw=1, label=f'Start: {losses_log[0]:.2f}')
    axes[0].axhline(losses_log[-1], color='green',  ls='--', lw=1, label=f'Final: {losses_log[-1]:.2f}')
    axes[0].set_title('Cross-Entropy Loss'); axes[0].set_xlabel('Steps'); axes[0].set_ylabel('Loss')
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

    # Plot 2: Perplexity (log scale)
    perp = [math.exp(min(l, 12)) for l in smooth_log]
    axes[1].plot(steps_log, perp, color='darkorchid', lw=1.5)
    axes[1].fill_between(steps_log, perp, alpha=0.15, color='darkorchid')
    axes[1].set_title('Perplexity (log scale)'); axes[1].set_xlabel('Steps'); axes[1].set_ylabel('PPL')
    axes[1].set_yscale('log'); axes[1].grid(True, alpha=0.3)
    axes[1].set_title(f'Perplexity  {perp[0]:.0f} → {perp[-1]:.0f}')

    # Plot 3: Per-segment loss drop (green = learning)
    N = len(losses_log)
    chunk = max(1, N // 30)
    seg_steps, seg_delta = [], []
    for i in range(chunk, N, chunk):
        seg_steps.append(steps_log[i])
        seg_delta.append(losses_log[i-chunk] - losses_log[i])
    colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in seg_delta]
    axes[2].bar(seg_steps, seg_delta, color=colors, alpha=0.8, width=seg_steps[0] if seg_steps else 1)
    axes[2].axhline(0, color='black', lw=0.8)
    axes[2].set_title('Per-Segment Loss Drop (🟢=learning)'); axes[2].set_xlabel('Steps')
    axes[2].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plot_path = '/kaggle/working/loss_curve.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved: {plot_path}')

    print(f'\n{"─"*45}')
    print(f'  Start  : {losses_log[0]:.4f}  PPL={math.exp(min(losses_log[0],12)):.0f}')
    print(f'  Final  : {losses_log[-1]:.4f}  PPL={math.exp(min(losses_log[-1],12)):.0f}')
    print(f'  Best   : {min(losses_log):.4f}  PPL={math.exp(min(min(losses_log),12)):.0f}')
    print(f'  Steps  : {steps_log[-1]}')
    print(f'  Drop   : {losses_log[0]-losses_log[-1]:.4f} nats')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 9 — Evaluate + Generate Text
# ═══════════════════════════════════════════════════════
os.chdir(WORK_DIR)
GPU_BIN = f'{WORK_DIR}/build/logos_gpu'
CPU_BIN = f'{WORK_DIR}/build/logos'
BINARY  = GPU_BIN if os.path.exists(GPU_BIN) else CPU_BIN

ckpts = sorted([f for f in glob.glob(f'{WORK_DIR}/*.bin') if 'vocab' not in f])
print(f'Checkpoints found: {[os.path.basename(c) for c in ckpts]}')

if ckpts:
    latest = ckpts[-1]
    print(f'\nUsing: {os.path.basename(latest)}')

    print('\n=== Evaluation (perplexity) ===')
    os.system(f'{BINARY} --eval {TRAIN_FILE} {latest} 2>&1')

    print('\n=== Text Generation ===')
    prompts = [
        'Once upon a time',
        'The little girl named',
        'Tom and his dog',
        'In a small village there',
        'She was very happy because',
    ]
    for p in prompts:
        print(f"\n{'─'*45}")
        print(f"Prompt: '{p}'")
        print('─'*45)
        os.system(f'{BINARY} --generate {latest} "{p}" 2>&1')
else:
    print('⚠️  No checkpoint found — run Cell 7 first')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 10 — Save All Outputs + Summary
# ═══════════════════════════════════════════════════════
os.makedirs(OUT_DIR, exist_ok=True)
saved = []

for f in glob.glob(f'{WORK_DIR}/*.bin'):
    shutil.copy(f, OUT_DIR); saved.append(os.path.basename(f))

for b in [f'{WORK_DIR}/build/logos', f'{WORK_DIR}/build/logos_gpu']:
    if os.path.exists(b):
        shutil.copy(b, OUT_DIR); saved.append(os.path.basename(b))

for src_f in [LOG_FILE, '/kaggle/working/loss_curve.png']:
    if os.path.exists(src_f):
        shutil.copy(src_f, OUT_DIR); saved.append(os.path.basename(src_f))

print(f'✅ Saved {len(saved)} files to {OUT_DIR}:')
os.system(f'ls -lh {OUT_DIR}')

print(f'\n{"═"*45}')
print('  LOGOS Training Summary')
print(f'  Dataset : 50MB TinyStories')
print(f'  Model   : d=128 | L=4 | H=8')
print(f'  Optim   : Langevin Dynamics')
if losses_log:
    print(f'  Loss    : {losses_log[0]:.3f} → {losses_log[-1]:.3f}')
    print(f'  PPL     : {math.exp(min(losses_log[0],12)):.0f} → {math.exp(min(losses_log[-1],12)):.0f}')
print(f'{"═"*45}')
print('\n📥 Download from Kaggle Output tab!')